# Mini GPT-2 with BPE Tokenisation

This is a **parallel version** of the character-level notebook, but using **Byte-Pair Encoding (BPE)** tokenisation instead.

## What's different?
- **Character-level**: Model sees individual characters (a, b, c, space, etc.)
- **BPE**: Model sees subword units ("patient", "ing", "##ed", etc.)

## Why BPE is better:
✓ Shorter sequences = faster training
✓ Better handles rare words
✓ Scales to large vocabularies
✓ Used by GPT-2, GPT-3, and modern LLMs

Everything else stays the same: model architecture, training loop, visualizations. Only tokenization changes.

## Setup and imports

In [ ]:
# Install tokenizers library (needed for BPE)
!pip install tokenizers --quiet

In [ ]:
import math
import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
import json
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# Reproducibility
seed = 42
torch.manual_seed(seed)

# Use GPU if available, else CPU
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

## 1. Load training data

Same as before — we'll use the Healthcare sample data.

In [ ]:
# Load the sample Healthcare data
with open('sample_healthcare_transformer_data.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print('Corpus length (characters):', len(text))
print('Sample preview:')
print(text[:500])

## 2. BPE Tokenization

This is where BPE differs from character-level tokenization.

**What BPE does:**
1. Starts with individual characters
2. Finds the most common pair of characters and merges them into a single token
3. Repeats until vocab size is reached
4. Result: vocabulary of subword tokens (like "ing", "tion", "patient")

**Benefits:**
- Words are broken into meaningful pieces
- Vocabulary stays bounded (you control vocab_size)
- Sequences are shorter → faster training

In [ ]:
# Save text to a temporary file (BPE trainer needs file input)
temp_train_file = 'temp_bpe_train.txt'
with open(temp_train_file, 'w', encoding='utf-8') as f:
    f.write(text)

# Create and train BPE tokenizer
tokenizer = Tokenizer(BPE(unk_token="<unk>"))

# Configure trainer
# vocab_size=500 means we'll have ~500 subword tokens instead of ~50 characters
trainer = BpeTrainer(
    vocab_size=500,
    special_tokens=["<unk>", "<pad>", "<eos>"]
)

# Use Whitespace pre-tokenizer (splits on spaces first)
tokenizer.pre_tokenizer = Whitespace()

# Train the tokenizer on our data
print("Training BPE tokenizer...")
tokenizer.train([temp_train_file], trainer)
print("BPE tokenizer trained!")

In [ ]:
# Get vocabulary size
vocab_size = tokenizer.get_vocab_size()
print(f'BPE Vocabulary size: {vocab_size}')

# Show some example tokens from the vocabulary
vocab = tokenizer.get_vocab()
print(f'\nFirst 30 tokens in BPE vocabulary:')
for i, (token, idx) in enumerate(sorted(vocab.items(), key=lambda x: x[1])[:30]):
    print(f"  {idx}: '{token}'")

In [ ]:
# Define encode and decode functions using BPE tokenizer
def encode(s):
    """Encode string to token IDs using BPE"""
    return tokenizer.encode(s).ids

def decode(token_ids):
    """Decode token IDs back to string"""
    return tokenizer.decode(token_ids)

# Test encoding/decoding
test_string = "Patient: I have a fever"
encoded = encode(test_string)
decoded = decode(encoded)

print(f'Original: {test_string}')
print(f'Encoded (tokens): {encoded}')
print(f'Decoded: {decoded}')
print(f'\nWith character-level tokenization: {len(test_string)} tokens')
print(f'With BPE tokenization: {len(encoded)} tokens')
print(f'Compression ratio: {len(test_string) / len(encoded):.2f}x')

In [ ]:
# Encode the entire training text
print("Encoding full text with BPE...")
encoded_text = encode(text)
data = torch.tensor(encoded_text, dtype=torch.long)

print(f'\nOriginal text length (characters): {len(text)}')
print(f'Encoded data length (tokens): {len(data)}')
print(f'Overall compression: {len(text) / len(data):.2f}x')
print(f'Data shape: {data.shape}')
print(f'Vocabulary size: {vocab_size}')

## 3. Train/validation split and batching

Same as before — the data is now BPE tokens instead of characters.

In [ ]:
# Hyperparameters
batch_size = 32
block_size = 64
max_iters = 1200
learning_rate = 3e-4
eval_interval = 100
eval_iters = 50
n_embd = 96
n_head = 4
n_layer = 3
dropout = 0.15

# Train/validation split
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f'Train data size: {len(train_data)} tokens')
print(f'Val data size: {len(val_data)} tokens')

# Function to generate batches
def get_batch(split='train'):
    source = train_data if split == 'train' else val_data
    ix = torch.randint(len(source) - block_size - 1, (batch_size,))
    x = torch.stack([source[i:i+block_size] for i in ix])
    y = torch.stack([source[i+1:i+block_size+1] for i in ix])
    return x.to(DEVICE), y.to(DEVICE)

# Test batch
x, y = get_batch('train')
print(f'\nBatch shape: x={x.shape}, y={y.shape}')

## 4. Model architecture

**IMPORTANT:** Only change from character-level version is the embedding dimension adjusted for BPE vocabulary size.
Everything else (attention heads, layers, normalization) is identical.

In [ ]:
class Head(nn.Module):
    """Single attention head"""
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        wei = q @ k.transpose(-2, -1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    """Multiple attention heads in parallel"""
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

class FeedForward(nn.Module):
    """Simple MLP feed-forward network"""
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """Transformer block: attention + feed-forward + layer norm"""
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class MiniGPT(nn.Module):
    """Mini GPT-2 style transformer"""
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=DEVICE))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# Create model
model = MiniGPT().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f'Model created. Total parameters: {total_params:,}')

## 5. Training loop

Identical to character-level version. We're training on BPE tokens now, but the training process is the same.

In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Training
train_losses = []
val_losses = []

print("Starting training...")
model.train()

for iter in range(max_iters):
    # Evaluate periodically
    if iter % eval_interval == 0:
        losses = estimate_loss()
        train_losses.append(losses['train'].item())
        val_losses.append(losses['val'].item())
        print(f"Step {iter:4d}: train loss = {losses['train']:.4f}, val loss = {losses['val']:.4f}")

    # Training step
    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print("Training complete!")
print(f'Final train loss: {train_losses[-1]:.4f}')
print(f'Final val loss: {val_losses[-1]:.4f}')

## 6. Loss visualization

Compare training vs validation loss over time.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(len(train_losses)), train_losses, label='Train Loss', marker='o', markersize=4)
plt.plot(range(len(val_losses)), val_losses, label='Validation Loss', marker='s', markersize=4)
plt.xlabel('Eval Step')
plt.ylabel('Loss')
plt.title('BPE Tokenization: Training vs Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Compare with character-level expectations
print("Training with BPE tokenization:")
print(f"  Initial train loss: {train_losses[0]:.4f}")
print(f"  Final train loss: {train_losses[-1]:.4f}")
print(f"  Loss reduction: {train_losses[0] - train_losses[-1]:.4f}")
print(f"\nNote: BPE uses subword tokens instead of characters.")
print(f"This should train faster (fewer tokens to predict) and potentially achieve better generalization.")

## 7. Text generation with sampling strategies

Same sampling strategies as the character-level version, but now generating BPE tokens.

In [ ]:
def generate_with_sampling(prompt, strategy='greedy', temperature=1.0, top_k=50, top_p=0.9, max_tokens=100):
    """Generate text with various sampling strategies"""
    model.eval()
    
    # Encode prompt
    prompt_tokens = encode(prompt)
    context = torch.tensor([prompt_tokens], dtype=torch.long, device=DEVICE)
    
    with torch.no_grad():
        for _ in range(max_tokens):
            # Get predictions
            context_cond = context[:, -block_size:]
            logits, _ = model(context_cond)
            logits = logits[:, -1, :] / temperature
            
            if strategy == 'greedy':
                next_token = torch.argmax(logits, dim=-1, keepdim=True)
            
            elif strategy == 'temperature':
                probs = F.softmax(logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            
            elif strategy == 'top_k':
                v, _ = torch.topk(logits, min(top_k, vocab_size))
                logits[logits < v[:, [-1]]] = float('-inf')
                probs = F.softmax(logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            
            elif strategy == 'top_p':
                probs = F.softmax(logits, dim=-1)
                sorted_probs, sorted_indices = torch.sort(probs, descending=True)
                cumsum_probs = torch.cumsum(sorted_probs, dim=-1)
                sorted_indices_to_remove = cumsum_probs > top_p
                sorted_indices_to_remove[..., 0] = False
                indices_to_remove = sorted_indices[sorted_indices_to_remove]
                logits[0, indices_to_remove] = float('-inf')
                probs = F.softmax(logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
            
            context = torch.cat([context, next_token], dim=1)
    
    return decode(context[0].tolist())

print("Text generation examples with BPE:")
print("="*60)
prompt = "Patient: I have a "

print("\n1. GREEDY (highest probability):")
print(generate_with_sampling(prompt, strategy='greedy', max_tokens=100))

print("\n2. TEMPERATURE 0.5 (more confident):")
print(generate_with_sampling(prompt, strategy='temperature', temperature=0.5, max_tokens=100))

print("\n3. TEMPERATURE 1.0 (normal):")
print(generate_with_sampling(prompt, strategy='temperature', temperature=1.0, max_tokens=100))

print("\n4. TEMPERATURE 1.5 (more creative):")
print(generate_with_sampling(prompt, strategy='temperature', temperature=1.5, max_tokens=100))

print("\n5. TOP-K 10 (keep top 10 tokens):")
print(generate_with_sampling(prompt, strategy='top_k', top_k=10, max_tokens=100))

print("\n6. TOP-P 0.9 (cumulative prob):")
print(generate_with_sampling(prompt, strategy='top_p', top_p=0.9, max_tokens=100))

## 8. Attention visualization

Extract and visualize attention patterns from the first head of the first block.

## 7b. Visualizing Sampling Methods

How do these sampling strategies actually work? Let's create a visual comparison showing how each method selects tokens from the same probability distribution.

**Key insight:** Each sampling method picks different tokens from the probability distribution in different ways:
- **Greedy**: Always pick the highest probability token
- **Temperature**: Reshape the distribution (flatten or sharpen it)
- **Top-K**: Only consider the K most likely tokens
- **Top-P (Nucleus)**: Consider tokens until cumulative probability reaches P


In [ ]:
import numpy as np

# Create a synthetic probability distribution to visualize with
# (imagine these are the model's predicted probabilities for next token)
np.random.seed(42)
num_tokens = 20
probs = np.random.exponential(0.5, num_tokens)
probs = probs / probs.sum()  # Normalize to probabilities

token_names = [f"Token {i}" for i in range(num_tokens)]

# Create figure with 5 subplots (one for each sampling method)
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('How Different Sampling Methods Select Tokens from Same Probability Distribution', 
             fontsize=14, fontweight='bold')

# 1. GREEDY - Pick the highest probability
ax = axes[0, 0]
colors = ['red' if i == np.argmax(probs) else 'lightblue' for i in range(num_tokens)]
ax.bar(range(num_tokens), probs, color=colors, edgecolor='black', linewidth=1.5)
ax.set_title('1. GREEDY\n(Always pick highest)', fontweight='bold', fontsize=11)
ax.set_ylabel('Probability')
ax.set_xlabel('Token Index')
ax.text(np.argmax(probs), np.max(probs)*1.05, '← SELECTED', ha='center', fontweight='bold', color='red')
ax.set_ylim(0, max(probs)*1.15)

# 2. TEMPERATURE 0.5 - Sharpen the distribution
ax = axes[0, 1]
temp_05 = probs ** (1/0.5)
temp_05 = temp_05 / temp_05.sum()
colors_temp05 = ['red' if i == np.argmax(temp_05) else 'lightgreen' for i in range(num_tokens)]
ax.bar(range(num_tokens), temp_05, color=colors_temp05, edgecolor='black', linewidth=1.5, alpha=0.7)
ax.bar(range(num_tokens), probs, color='none', edgecolor='gray', linewidth=1, linestyle='--', label='Original')
ax.set_title('2. TEMPERATURE 0.5\n(Sharp: More confident)', fontweight='bold', fontsize=11)
ax.set_ylabel('Probability')
ax.set_xlabel('Token Index')
ax.legend(['Original'], loc='upper right')
ax.set_ylim(0, max(temp_05)*1.15)

# 3. TEMPERATURE 1.5 - Flatten the distribution
ax = axes[0, 2]
temp_15 = probs ** (1/1.5)
temp_15 = temp_15 / temp_15.sum()
colors_temp15 = ['orange' if i == np.argmax(temp_15) else 'lightyellow' for i in range(num_tokens)]
ax.bar(range(num_tokens), temp_15, color=colors_temp15, edgecolor='black', linewidth=1.5, alpha=0.7)
ax.bar(range(num_tokens), probs, color='none', edgecolor='gray', linewidth=1, linestyle='--', label='Original')
ax.set_title('3. TEMPERATURE 1.5\n(Flat: More creative)', fontweight='bold', fontsize=11)
ax.set_ylabel('Probability')
ax.set_xlabel('Token Index')
ax.legend(['Original'], loc='upper right')
ax.set_ylim(0, max(temp_15)*1.15)

# 4. TOP-K=5 - Keep only top 5 tokens
ax = axes[1, 0]
top_k = 5
sorted_indices = np.argsort(-probs)[:top_k]
colors_topk = ['purple' if i in sorted_indices else 'lightgray' for i in range(num_tokens)]
bars = ax.bar(range(num_tokens), probs, color=colors_topk, edgecolor='black', linewidth=1.5)
ax.set_title(f'4. TOP-K (K=5)\n(Keep only top {top_k} tokens)', fontweight='bold', fontsize=11)
ax.set_ylabel('Probability')
ax.set_xlabel('Token Index')
ax.axhline(y=np.sort(probs)[-top_k], color='purple', linestyle='--', linewidth=2, label='Cutoff')
ax.legend()
ax.set_ylim(0, max(probs)*1.15)

# 5. TOP-P=0.9 - Keep tokens until cumulative prob reaches 0.9
ax = axes[1, 1]
top_p = 0.9
sorted_indices_p = np.argsort(-probs)
cumsum = np.cumsum(probs[sorted_indices_p])
cutoff_idx = np.where(cumsum >= top_p)[0][0]
selected_p = sorted_indices_p[:cutoff_idx+1]
colors_topp = ['green' if i in selected_p else 'lightgray' for i in range(num_tokens)]
ax.bar(range(num_tokens), probs, color=colors_topp, edgecolor='black', linewidth=1.5)
ax.set_title(f'5. TOP-P (P=0.9)\n(Until cumulative prob = {top_p})', fontweight='bold', fontsize=11)
ax.set_ylabel('Probability')
ax.set_xlabel('Token Index')
ax.text(0.5, 0.95, f'Selected {cutoff_idx+1} tokens\nCumulative prob: {cumsum[cutoff_idx]:.2f}', 
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))
ax.set_ylim(0, max(probs)*1.15)

# 6. Comparison table
ax = axes[1, 2]
ax.axis('off')
comparison_text = """
SAMPLING METHOD COMPARISON

🔴 GREEDY
  → Always pick highest
  → Deterministic
  → Repetitive output
  → Best for: Consistency

🟢 TEMPERATURE
  → Reshape distribution
  → 0.5 = Sharp (confident)
  → 1.5 = Flat (creative)
  → Best for: Balance

🟣 TOP-K (K=5-50)
  → Keep top K tokens only
  → Prevents very unlikely tokens
  → Controls diversity
  → Best for: Reasonable variety

🟢 TOP-P (P=0.9)
  → Keep until cumsum=P
  → Adaptive to distribution
  → Professional quality
  → Best for: Production use
"""
ax.text(0.05, 0.95, comparison_text, transform=ax.transAxes, fontsize=9,
        verticalalignment='top', family='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("SAMPLING METHODS EXPLAINED")
print("="*70)
print(f"\n📊 Probability Distribution Summary:")
print(f"   Highest probability: {np.max(probs):.4f} (Token {np.argmax(probs)})")
print(f"   Top 5 tokens account for: {np.sum(np.sort(probs)[-5:]):.1%} of probability")
print(f"   Top 10 tokens account for: {np.sum(np.sort(probs)[-10:]):.1%} of probability")
print(f"\n🎯 What each method does:")
print(f"   • GREEDY: Selects Token {np.argmax(probs)} (p={np.max(probs):.4f})")
print(f"   • TEMPERATURE 0.5: Sharpens distribution (even more confident)")
print(f"   • TEMPERATURE 1.5: Flattens distribution (more creative/random)")
print(f"   • TOP-K (5): Selects from top 5 tokens only")
print(f"   • TOP-P (0.9): Selects until cumulative probability reaches 90%")

In [ ]:
# Let's also show an ANIMATED example with the model's real probabilities
print("\n" + "="*70)
print("REAL EXAMPLE: Model predicting next token after 'Patient: Doctor'")
print("="*70)

# Get the model's actual probability distribution for the next token
model.eval()
with torch.no_grad():
    test_prompt = "Patient: I have a h"
    test_tokens = encode(test_prompt)
    test_input = torch.tensor([test_tokens], dtype=torch.long, device=DEVICE)
    
    logits, _ = model(test_input)
    # Get logits from the last position
    last_logits = logits[0, -1, :].cpu()
    real_probs = F.softmax(torch.tensor(last_logits), dim=-1).numpy()

# Find top tokens
top_indices = np.argsort(-real_probs)[:10]
top_probs = real_probs[top_indices]

# Create visualization of model's real predictions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Bar chart of top 10 tokens
token_strs = [tokenizer.decode([idx]) for idx in top_indices]
colors_real = ['red' if i == 0 else 'lightblue' for i in range(len(top_indices))]

ax1.barh(range(len(top_indices)), top_probs, color=colors_real, edgecolor='black', linewidth=1.5)
ax1.set_yticks(range(len(top_indices)))
ax1.set_yticklabels([f"{token_strs[i]!r} (ID:{top_indices[i]})" for i in range(len(top_indices))])
ax1.set_xlabel('Probability')
ax1.set_title(f"Top 10 Predicted Tokens After '{test_prompt}'", fontweight='bold', fontsize=12)
ax1.invert_yaxis()

for i, prob in enumerate(top_probs):
    ax1.text(prob, i, f' {prob:.3f}', va='center', fontweight='bold')

# Right: Show what each method would select
ax2.axis('off')

# Calculate selections for different methods
greedy_token = top_indices[0]
top_k_tokens = top_indices[:5]
top_p_threshold = 0.9
cumsum_probs = np.cumsum(top_probs)
top_p_tokens = top_indices[:np.where(cumsum_probs >= top_p_threshold)[0][0] + 1]

method_text = f"""
🎯 WHAT EACH METHOD WOULD SELECT:

Prompt: "{test_prompt}"

1️⃣  GREEDY
   → Token: {token_strs[0]!r}
   → Probability: {top_probs[0]:.4f} (100% confident)

2️⃣  TEMPERATURE 0.5
   → More confident version of greedy
   → Still likely: {token_strs[0]!r}

3️⃣  TEMPERATURE 1.5
   → Could pick any of: {token_strs[0]!r}, {token_strs[1]!r}, {token_strs[2]!r}...
   → More creative/varied

4️⃣  TOP-K (K=5)
   → Can pick from: {token_strs[0]!r}, {token_strs[1]!r}, {token_strs[2]!r}, ...
   → Prevents weird tokens
   → Still varied

5️⃣  TOP-P (P=0.9)
   → Cumulative prob ≥ 0.9:
   → {len(top_p_tokens)} tokens selected
   → Balances quality & diversity
   → RECOMMENDED for production! ✨

💡 KEY INSIGHT:
All methods use the SAME probability
distribution, but select differently!
"""

ax2.text(0.05, 0.95, method_text, transform=ax2.transAxes, fontsize=10,
        verticalalignment='top', family='monospace',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9, pad=1))

plt.tight_layout()
plt.show()

print(f"\n Visualization complete!")
print(f"   Model vocab size: {vocab_size}")
print(f"   Top token: {token_strs[0]!r} (p={top_probs[0]:.4f})")

In [ ]:
# Create a short prompt to visualize attention
short_prompt = "Patient: Doctor"
prompt_tokens = encode(short_prompt)
print(f"Prompt: {short_prompt}")
print(f"Tokens: {prompt_tokens}")
print(f"Number of tokens: {len(prompt_tokens)}")

In [ ]:
# Extract attention weights (hook into the first Head)
attention_weights = None

def hook_fn(module, input, output):
    global attention_weights
    # The attention weights are computed in the Head forward pass
    # We need to compute them again from the forward pass
    pass

# For simplicity, we'll manually compute attention for visualization
model.eval()
with torch.no_grad():
    idx_input = torch.tensor([prompt_tokens], dtype=torch.long, device=DEVICE)
    B, T = idx_input.shape
    
    # Get embeddings
    tok_emb = model.token_embedding_table(idx_input)
    pos_emb = model.position_embedding_table(torch.arange(T, device=DEVICE))
    x = tok_emb + pos_emb
    
    # Get attention from first block's first head
    first_block = model.blocks[0]
    first_head = first_block.sa.heads[0]
    
    # Compute attention manually
    k = first_head.key(x)
    q = first_head.query(x)
    v = first_head.value(x)
    
    head_size = q.shape[-1]
    wei = q @ k.transpose(-2, -1) * head_size**-0.5
    wei = wei.masked_fill(first_head.tril[:T, :T] == 0, float('-inf'))
    wei = F.softmax(wei, dim=-1)
    
    attention_map = wei[0].cpu().numpy()

# Visualize
plt.figure(figsize=(8, 6))
plt.imshow(attention_map, cmap='hot', aspect='auto')
plt.colorbar(label='Attention Weight')
plt.xlabel('Key Position (what to attend to)')
plt.ylabel('Query Position (where from)')
plt.title(f'First Attention Head: "{short_prompt}"')
plt.xticks(range(len(prompt_tokens)))
plt.yticks(range(len(prompt_tokens)))
plt.tight_layout()
plt.show()

## 9. Save model and tokenizer for later use

We save:
- The trained model weights
- The BPE tokenizer (crucial for inference!)
- Hyperparameters for reference

In [ ]:
# Save the trained model
torch.save(model, 'mini_gpt_bpe_model.pt')
print('✓ Model saved to mini_gpt_bpe_model.pt')

# Save the BPE tokenizer
tokenizer.save('mini_gpt_bpe_tokenizer.json')
print('✓ BPE Tokenizer saved to mini_gpt_bpe_tokenizer.json')

# Save hyperparameters
hparams = {
    'vocab_size': vocab_size,
    'block_size': block_size,
    'n_embd': n_embd,
    'n_head': n_head,
    'n_layer': n_layer,
    'dropout': dropout,
    'tokenization': 'BPE',
    'bpe_vocab_size': 500,
}

with open('mini_gpt_bpe_hparams.json', 'w') as f:
    json.dump(hparams, f, indent=2)
print('✓ Hyperparameters saved to mini_gpt_bpe_hparams.json')

## 10. Comparison: BPE vs Character-Level

### Why BPE is better for GPT-style models:

| Aspect | Character-Level | BPE |
|--------|-----------------|-----|
| **Vocab size** | ~50 unique chars | ~500 subword tokens |
| **Sequence length** | Very long | Shorter (fewer tokens) |
| **Training speed** | Slower | Faster |
| **Memory usage** | More tokens to process | Fewer tokens |
| **Out-of-vocab** | Never happens | Rare with BPE |
| **Semantic units** | Individual chars | Meaningful subwords |
| **Used by** | Simple demos | GPT-2, GPT-3, LLaMA |

### When to use each:
- **Character-level**: Learning/demos, very small vocabularies, single language
- **BPE**: Production models, multi-language support, large vocabularies, fast inference